In [1]:
import numpy as np
import pandas as pd

In [4]:
datos = pd.DataFrame({
    'hours_sleep':[1,2,3,4,5,6,7],
    'attendance':[60,65,70,75,80,85,87],
    'grades':[50,65,70,75,np.nan,85,np.nan]
})

datos

,hours_sleep,attendance,grades
0,1,60,50.0
1,2,65,65.0
2,3,70,70.0
3,4,75,75.0
4,5,80,NaN
5,6,85,85.0
6,7,87,NaN


In [7]:
from sklearn.linear_model import LinearRegression

In [15]:
train_df = datos[datos['grades'].notnull()]
train_df
missing_df = datos[datos['grades'].isnull()]
missing_df
x_train = train_df[['hours_sleep', 'attendance']]
y_train = train_df['grades']

#fit regression
modelo = LinearRegression()
modelo.fit(x_train,y_train)

x_test = missing_df[['hours_sleep', 'attendance']]
predict = modelo.predict(x_test)
predict


array([80.67567568, 89.90644491])

In [ ]:
# single imputation
data_filled = datos.copy()
data_filled.loc[data_filled['grades'].isnull(), 'grades'] = predict
data_filled

,hours_sleep,attendance,grades
0,1,60,50.000000
1,2,65,65.000000
2,3,70,70.000000
3,4,75,75.000000
4,5,80,80.675676
5,6,85,85.000000
6,7,87,89.906445


In [18]:
#bayesian regression
from sklearn.linear_model import BayesianRidge

In [25]:
train_df = datos[datos['grades'].notnull()]
missing_df = datos[datos['grades'].isnull()]
x_train = train_df[['hours_sleep', 'attendance']]
y_train = train_df['grades']

bayesian_model = BayesianRidge()
bayesian_model.fit(x_train, y_train)

x_test = missing_df[['hours_sleep', 'attendance']]
predict2 = bayesian_model.predict(x_test)
predict2

#computing the uncertanties
predict2, std = bayesian_model.predict(x_test, return_std=True)

# print(predict2 , std)
for predict, uncert in zip(predict2, std):
    print('predict', predict)
    print('uncertanty', uncert)


predict 80.45396941441557
uncertanty 3.789951707447654
predict 89.50945805401761
uncertanty 4.529439480000733


In [51]:
# challenge:
# generate differente values for imputation

aleatorios1 = np.random.normal(loc=predict2[0], scale=std.iloc[0], size=100)
aleatorios2 = np.random.normal(loc=predict2[1], scale=std.iloc[1], size=100)

-------

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import BayesianRidge

In [2]:
datos = pd.DataFrame({
    'hours_sleep':[1,2,3,4,5,6,7],
    'attendance':[60,65,70,75,80,85,87],
    'grades':[50,65,70,75,np.nan,85,np.nan]
})

datos

,hours_sleep,attendance,grades
0,1,60,50.0
1,2,65,65.0
2,3,70,70.0
3,4,75,75.0
4,5,80,NaN
5,6,85,85.0
6,7,87,NaN


In [3]:
train_df = datos[datos['grades'].notnull()]
train_df
missing_df = datos[datos['grades'].isnull()]
missing_df
x_train = train_df[['hours_sleep', 'attendance']]
y_train = train_df['grades']

In [4]:
bayesian_model = BayesianRidge()
bayesian_model.fit(x_train, y_train)

x_test = missing_df[['hours_sleep', 'attendance']]
predict2 = bayesian_model.predict(x_test)
predict2

#computing the uncertanties
predict2, std = bayesian_model.predict(x_test, return_std=True)

# print(predict2 , std)
for predict, uncert in zip(predict2, std):
    print('predict', predict)
    print('uncertanty', uncert)

predict 80.45396941441557
uncertanty 3.789951707447654
predict 89.50945805401761
uncertanty 4.529439480000733


In [33]:
m = 3
aleatorios1 = np.random.normal(loc=predict2[0], scale=std.iloc[0], size=m)
aleatorios2 = np.random.normal(loc=predict2[1], scale=std.iloc[1], size=m)
print(aleatorios1)
print(aleatorios2)

[74.1325346  83.31773781 79.07527288]
[97.33897278 92.50457872 92.73087039]


In [34]:
df1 = datos.copy()
df1.loc[df1['grades'].isnull(), 'grades'] = [aleatorios1[0], aleatorios2[0]]
df1

,hours_sleep,attendance,grades
0,1,60,50.000000
1,2,65,65.000000
2,3,70,70.000000
3,4,75,75.000000
4,5,80,74.132535
5,6,85,85.000000
6,7,87,97.338973


In [35]:
def mult_imputation(data, m):
    dfs = {}
    for j in range(m):
        dfm = data.copy()
        dfm.loc[dfm['grades'].isnull(), 'grades'] = [aleatorios1[j], aleatorios2[j]]
        dfs[f'df_{j}'] = dfm
    return dfs

ds = mult_imputation(datos, m)
ds

{'df_0':    hours_sleep  attendance     grades
 0            1          60  50.000000
 1            2          65  65.000000
 2            3          70  70.000000
 3            4          75  75.000000
 4            5          80  74.132535
 5            6          85  85.000000
 6            7          87  97.338973,
 'df_1':    hours_sleep  attendance     grades
 0            1          60  50.000000
 1            2          65  65.000000
 2            3          70  70.000000
 3            4          75  75.000000
 4            5          80  83.317738
 5            6          85  85.000000
 6            7          87  92.504579,
 'df_2':    hours_sleep  attendance     grades
 0            1          60  50.000000
 1            2          65  65.000000
 2            3          70  70.000000
 3            4          75  75.000000
 4            5          80  79.075273
 5            6          85  85.000000
 6            7          87  92.730870}

------

### compute within and between quantities for the last example and interpret.


### what is the objective of bayessian regression and the difference with "usual" linear regression
### what we are computing in bayessian regression?
- bayessian regression
- posterior distribution in bayessian regression
- conditional expected value in bayessian regression
- linear regression. expected value for impoutation


In [36]:
within_sample_mean = []
for i in range(m):
    prom = float(ds[f'df_{i}']['grades'].mean())
    within_sample_mean.append(prom)

within_sample_mean = np.array(within_sample_mean)
within_sample_mean


array([73.78164391, 74.40318807, 73.82944904])

In [37]:
pool_mean = float(np.sum(within_sample_mean/m))
pool_mean

74.00476034158413

In [38]:
within_sample_variance = []
for i in range(m):
    variance = float(ds[f'df_{i}']['grades'].var())
    within_sample_variance.append(variance)

within_sample_variance = np.array(within_sample_variance)
within_sample_variance

array([223.23191005, 203.85029117, 195.23338115])

In [30]:
between_imputation = np.sum((within_sample_mean-pool_mean)**2) / (m-1)
between_imputation



np.float64(3.1814239285571846)

In [32]:
total_variance = np.mean(within_sample_variance) + (1+1/m)*between_imputation
total_variance

np.float64(195.4969031282012)